# Module 8 • Large Language Models

# Lesson 50 • LLM Course Capstone — Building an End-to-End Intelligent NLP Application

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Advanced Capstone  
**Execution target:** CPU only

## Capstone Goal

Build a complete offline **NLP Research Assistant** that integrates:

- intent routing;
- prompt templates;
- retrieval-augmented generation;
- hybrid retrieval;
- structured outputs;
- tool use;
- citations;
- confidence and abstention;
- multilingual/Arabic handling;
- end-to-end evaluation.

The system is deliberately deterministic and offline so the full notebook runs without
API keys, internet access, or large model downloads.

## Learning Objectives

After completing this lesson, the learner should be able to:

- design an end-to-end LLM-era NLP architecture;
- route user requests by intent;
- index and retrieve external evidence;
- combine sparse and latent-semantic relevance signals;
- execute structured tools;
- assemble grounded answers with citations;
- return JSON output;
- estimate confidence and abstain;
- evaluate retrieval, grounding, and task success;
- support selected Arabic queries while preserving tashkeel.

## Table of Contents

1. Architecture  
2. Knowledge Base  
3. Normalization and Arabic  
4. Chunking  
5. Sparse and Dense-Like Retrieval  
6. Hybrid Retrieval  
7. Prompt Templates  
8. Intent Routing  
9. Tools  
10. Grounded Generation  
11. Confidence and Abstention  
12. End-to-End Assistant  
13. Demonstrations  
14. Evaluation Dataset  
15. Intent Accuracy  
16. Recall@k and MRR  
17. Task Success  
18. Groundedness and Citation Validity  
19. Robustness  
20. Ablation  
21. Error Taxonomy  
22. System Card  
23. Deployment and Safety  
24. Knowledge Check  
25. Exercises  
26. Course Summary

# 1. Architecture

```text
User Query
    |
    v
Intent Router
    |
    +---- Calculator
    |
    +---- Structured Output
    |
    +---- Retrieval
              |
              v
        Hybrid Retriever
              |
              v
        Grounded Answer
              |
              v
      Citation / Confidence
              |
              v
          Final Response
```

In [ ]:
import json
import platform
import re
from collections import Counter
from dataclasses import dataclass, field
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import Normalizer

SEED = 42
np.random.seed(SEED)

# 2. Knowledge Base

In [ ]:
documents = [
    {
        "document_id": "doc_transformer",
        "title": "Transformer Architecture",
        "language": "en",
        "topic": "transformers",
        "text": (
            "Transformers use self-attention to model token relationships. "
            "Decoder-only transformers use causal masking so future tokens remain hidden. "
            "Encoder models process input bidirectionally."
        ),
    },
    {
        "document_id": "doc_tokenization",
        "title": "Tokenization",
        "language": "en",
        "topic": "tokenization",
        "text": (
            "Tokenization converts text into units processed by a language model. "
            "Subword tokenization helps represent rare words and morphology. "
            "Tokenizer efficiency affects effective context length."
        ),
    },
    {
        "document_id": "doc_rag",
        "title": "Retrieval-Augmented Generation",
        "language": "en",
        "topic": "rag",
        "text": (
            "Retrieval-augmented generation retrieves external evidence before generation. "
            "The retriever selects relevant chunks and the generator uses them as context. "
            "Grounded answers should be supported by retrieved evidence."
        ),
    },
    {
        "document_id": "doc_metrics",
        "title": "RAG Evaluation",
        "language": "en",
        "topic": "evaluation",
        "text": (
            "Recall at k measures whether relevant evidence appears in the top k retrieved results. "
            "Precision at k measures the relevant fraction of retrieved results. "
            "Mean reciprocal rank rewards placing the first relevant result near the top."
        ),
    },
    {
        "document_id": "doc_lora",
        "title": "LoRA",
        "language": "en",
        "topic": "peft",
        "text": (
            "LoRA is a parameter-efficient fine-tuning method. "
            "It keeps base model weights frozen and learns low-rank update matrices. "
            "LoRA reduces the number of trainable parameters compared with full fine-tuning."
        ),
    },
    {
        "document_id": "doc_agents",
        "title": "LLM Agents",
        "language": "en",
        "topic": "agents",
        "text": (
            "An LLM agent combines planning, tool selection, tool execution, observations, and stopping rules. "
            "Tool schemas define valid function names and arguments. "
            "Agent traces help diagnose failures."
        ),
    },
    {
        "document_id": "doc_multimodal",
        "title": "Multimodal Models",
        "language": "en",
        "topic": "multimodal",
        "text": (
            "Multimodal language models combine text with modalities such as images. "
            "Vision encoders produce visual representations that can be projected into a language-model-compatible space. "
            "Multimodal evaluation should test hallucination and cross-modal grounding."
        ),
    },
    {
        "document_id": "doc_arabic",
        "title": "Arabic NLP",
        "language": "ar",
        "topic": "arabic",
        "text": (
            "Arabic has rich morphology and attached clitics. "
            "For fully vocalized Arabic tasks, tashkeel should be preserved consistently. "
            "Evaluation should distinguish vocalized and unvocalized text."
        ),
    },
]

pd.DataFrame(documents)[["document_id", "title", "language", "topic"]]

# 3. Normalization and Arabic

Arabic is kept as Unicode. The pipeline does **not** strip tashkeel.

In [ ]:
WORD_PATTERN = re.compile(r"\b\w+(?:[-']\w+)*\b", flags=re.UNICODE)
ARABIC_DIACRITICS = set("\u064b\u064c\u064d\u064e\u064f\u0650\u0651\u0652")

def normalize_text(text):
    return " ".join(WORD_PATTERN.findall(text.lower()))

def contains_tashkeel(text):
    return any(ch in ARABIC_DIACRITICS for ch in text)

print(normalize_text("Retrieval-Augmented Generation (RAG)!"))
print(contains_tashkeel("وَسَيَكْتُبُونَهَا"))

For fully vocalized Arabic tasks, exact diacritics should be preserved in the source,
query, output, and evaluation.

# 4. Chunking

In [ ]:
SENTENCE_PATTERN = re.compile(r"(?<=[.!?])\s+")

def split_sentences(text):
    return [s.strip() for s in SENTENCE_PATTERN.split(text.strip()) if s.strip()]

def build_chunks(documents, sentences_per_chunk=2, overlap=1):
    rows = []
    stride = sentences_per_chunk - overlap
    for doc in documents:
        sentences = split_sentences(doc["text"])
        chunk_number = 0
        for start in range(0, len(sentences), stride):
            selected = sentences[start:start + sentences_per_chunk]
            if not selected:
                continue
            rows.append({
                "chunk_id": f"{doc['document_id']}_chunk_{chunk_number:02d}",
                "document_id": doc["document_id"],
                "title": doc["title"],
                "language": doc["language"],
                "topic": doc["topic"],
                "text": " ".join(selected),
            })
            chunk_number += 1
            if start + sentences_per_chunk >= len(sentences):
                break
    return pd.DataFrame(rows)

chunk_frame = build_chunks(documents)
chunk_frame.head()

# 5. Sparse and Dense-Like Retrieval

TF-IDF supplies lexical matching. Truncated SVD creates a compact latent-semantic
projection for the offline dense-like demonstration.

In [ ]:
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
sparse_matrix = vectorizer.fit_transform(chunk_frame["text"])

n_components = min(
    12,
    sparse_matrix.shape[0] - 1,
    sparse_matrix.shape[1] - 1,
)

svd = TruncatedSVD(n_components=n_components, random_state=SEED)
dense_matrix = svd.fit_transform(sparse_matrix)
normalizer = Normalizer()
dense_matrix = normalizer.fit_transform(dense_matrix)

print("Sparse:", sparse_matrix.shape)
print("Dense-like:", dense_matrix.shape)

In [ ]:
def minmax(values):
    values = np.asarray(values, dtype=float)
    lo, hi = values.min(), values.max()
    if hi - lo < 1e-12:
        return np.zeros_like(values)
    return (values - lo) / (hi - lo)

def sparse_scores(query):
    q = vectorizer.transform([query])
    return cosine_similarity(q, sparse_matrix)[0]

def dense_scores(query):
    q_sparse = vectorizer.transform([query])
    q_dense = normalizer.transform(svd.transform(q_sparse))
    return cosine_similarity(q_dense, dense_matrix)[0]

def hybrid_scores(query, sparse_weight=0.55):
    return (
        sparse_weight * minmax(sparse_scores(query))
        + (1.0 - sparse_weight) * minmax(dense_scores(query))
    )

# 6. Hybrid Retrieval

In [ ]:
def retrieve(query, top_k=3, language=None, sparse_weight=0.55):
    scores = hybrid_scores(query, sparse_weight=sparse_weight).copy()
    if language is not None:
        mask = chunk_frame["language"].to_numpy() == language
        scores[~mask] = -np.inf

    ranked = np.argsort(scores)[::-1]
    rows = []
    for idx in ranked:
        if not np.isfinite(scores[idx]):
            continue
        row = chunk_frame.iloc[int(idx)]
        rows.append({
            "rank": len(rows) + 1,
            "score": float(scores[idx]),
            "chunk_id": row["chunk_id"],
            "document_id": row["document_id"],
            "title": row["title"],
            "language": row["language"],
            "topic": row["topic"],
            "text": row["text"],
        })
        if len(rows) >= top_k:
            break
    return pd.DataFrame(rows)

retrieve("What does recall at k measure?", top_k=3)

# 7. Prompt Templates

In [ ]:
QA_PROMPT = (
    "You are a grounded NLP research assistant.\n\n"
    "Question:\n{question}\n\n"
    "Evidence:\n{context}\n\n"
    "Use only the evidence. If evidence is insufficient, abstain. "
    "Preserve source identifiers."
)

def build_prompt(question, context):
    return QA_PROMPT.format(question=question, context=context)

# 8. Intent Routing

In [ ]:
def detect_intent(query):
    q = query.lower()

    if re.search(r"\d+\s*[+\-*/]\s*\d+", q):
        return "calculate"

    if "json" in q or "structured" in q:
        return "structured"

    terms = [
        "transformer", "rag", "retrieval", "tokenization", "lora",
        "agent", "multimodal", "tashkeel", "arabic", "recall", "mrr",
    ]
    if any(term in q for term in terms):
        return "retrieve"

    # Arabic cue terms for the controlled capstone.
    if any(term in query for term in ["التَّشْكِيل", "الْعَرَبِيَّة", "عَرَبِي"]):
        return "retrieve"

    return "unknown"

pd.Series({
    "What is LoRA?": detect_intent("What is LoRA?"),
    "Calculate 4 + 8": detect_intent("Calculate 4 + 8"),
    "Return JSON: What is RAG?": detect_intent("Return JSON: What is RAG?"),
})

# 9. Tools

In [ ]:
SAFE_OPERATORS = {
    "+": lambda a, b: a + b,
    "-": lambda a, b: a - b,
    "*": lambda a, b: a * b,
    "/": lambda a, b: a / b,
}

def extract_expression(query):
    match = re.search(
        r"(-?\d+(?:\.\d+)?)\s*([+\-*/])\s*(-?\d+(?:\.\d+)?)",
        query,
    )
    return None if match is None else " ".join(match.groups())

def calculator(expression):
    match = re.fullmatch(
        r"\s*(-?\d+(?:\.\d+)?)\s*([+\-*/])\s*(-?\d+(?:\.\d+)?)\s*",
        expression,
    )
    if not match:
        return {"ok": False, "error": "unsupported expression"}

    left, symbol, right = match.groups()
    left, right = float(left), float(right)

    if symbol == "/" and right == 0:
        return {"ok": False, "error": "division by zero"}

    return {"ok": True, "result": SAFE_OPERATORS[symbol](left, right)}

calculator("24 * 6")

# 10. Grounded Generation

In [ ]:
def sentence_score(query, sentence):
    q_terms = set(normalize_text(query).split())
    s_terms = set(normalize_text(sentence).split())
    if not q_terms:
        return 0.0
    return len(q_terms & s_terms) / len(q_terms)

def grounded_generate(query, retrieved, minimum_score=0.08):
    candidates = []

    for row in retrieved.itertuples(index=False):
        for sentence in split_sentences(row.text):
            candidates.append({
                "score": sentence_score(query, sentence),
                "sentence": sentence,
                "chunk_id": row.chunk_id,
            })

    if not candidates:
        return (
            "The available evidence is insufficient to answer this question.",
            [],
            0.0,
        )

    best = max(candidates, key=lambda item: item["score"])

    if best["score"] < minimum_score:
        return (
            "The available evidence is insufficient to answer this question.",
            [],
            best["score"],
        )

    return best["sentence"], [best["chunk_id"]], best["score"]

# 11. Confidence and Abstention

In [ ]:
ABSTENTION_THRESHOLD = 0.22

def estimate_confidence(retrieved, evidence_score):
    if len(retrieved) == 0:
        return 0.0
    retrieval_score = float(retrieved.iloc[0]["score"])
    return float(np.clip(
        0.65 * retrieval_score + 0.35 * evidence_score,
        0.0,
        1.0,
    ))

# 12. End-to-End Assistant

In [ ]:
@dataclass
class AssistantResponse:
    answer: str
    intent: str
    confidence: float
    citations: list[str] = field(default_factory=list)
    abstained: bool = False
    structured: dict[str, Any] | None = None

def detect_language(query):
    return "ar" if any("\u0600" <= ch <= "\u06ff" for ch in query) else "en"

def assistant_app(query):
    intent = detect_intent(query)

    if intent == "calculate":
        expression = extract_expression(query)
        result = calculator(expression or "")
        if result.get("ok"):
            return AssistantResponse(
                answer=str(result["result"]),
                intent=intent,
                confidence=1.0,
            )
        return AssistantResponse(
            answer="The calculation could not be completed.",
            intent=intent,
            confidence=0.0,
            abstained=True,
        )

    if intent in {"retrieve", "structured"}:
        language = detect_language(query)
        retrieved = retrieve(
            query,
            top_k=3,
            language="ar" if language == "ar" else None,
        )

        answer, citations, evidence_score = grounded_generate(query, retrieved)
        confidence = estimate_confidence(retrieved, evidence_score)
        abstained = confidence < ABSTENTION_THRESHOLD or not citations

        if abstained:
            answer = "The available evidence is insufficient to answer this question."
            citations = []

        if intent == "structured":
            structured = {
                "answer": answer,
                "intent": intent,
                "confidence": round(confidence, 4),
                "citations": citations,
                "abstained": abstained,
            }
            return AssistantResponse(
                answer=json.dumps(structured, ensure_ascii=False),
                intent=intent,
                confidence=confidence,
                citations=citations,
                abstained=abstained,
                structured=structured,
            )

        return AssistantResponse(
            answer=answer,
            intent=intent,
            confidence=confidence,
            citations=citations,
            abstained=abstained,
        )

    return AssistantResponse(
        answer="No supported capability matches this request.",
        intent="unknown",
        confidence=0.0,
        abstained=True,
    )

# 13. Demonstrations

In [ ]:
demo_queries = [
    "What is LoRA?",
    "What does recall at k measure?",
    "Calculate 24 * 6.",
    "Return JSON: What is RAG?",
    "مَاذَا يَجِبُ أَنْ نَفْعَلَ بِالتَّشْكِيلِ فِي مَهَامِّ الْعَرَبِيَّةِ؟",
]

demo_rows = []
for query in demo_queries:
    response = assistant_app(query)
    demo_rows.append({
        "query": query,
        "intent": response.intent,
        "answer": response.answer,
        "confidence": response.confidence,
        "citations": response.citations,
        "abstained": response.abstained,
    })

pd.DataFrame(demo_rows)

# 14. Evaluation Dataset

In [ ]:
evaluation_cases = [
    {
        "query": "What is LoRA?",
        "intent": "retrieve",
        "relevant_document": "doc_lora",
        "must_contain": "parameter",
    },
    {
        "query": "What does recall at k measure?",
        "intent": "retrieve",
        "relevant_document": "doc_metrics",
        "must_contain": "relevant",
    },
    {
        "query": "What is an LLM agent?",
        "intent": "retrieve",
        "relevant_document": "doc_agents",
        "must_contain": "tool",
    },
    {
        "query": "Calculate 9 + 4.",
        "intent": "calculate",
        "relevant_document": None,
        "must_contain": "13",
    },
    {
        "query": "Return JSON: What is RAG?",
        "intent": "structured",
        "relevant_document": "doc_rag",
        "must_contain": "answer",
    },
    {
        "query": "What is the capital of Mars?",
        "intent": "unknown",
        "relevant_document": None,
        "must_contain": "No supported",
    },
]

pd.DataFrame(evaluation_cases)

# 15. Intent Accuracy

In [ ]:
intent_eval = pd.DataFrame([
    {
        "query": item["query"],
        "gold": item["intent"],
        "prediction": detect_intent(item["query"]),
    }
    for item in evaluation_cases
])
intent_eval["correct"] = intent_eval["gold"] == intent_eval["prediction"]
intent_accuracy = float(intent_eval["correct"].mean())
intent_eval

# 16. Recall@k and MRR

In [ ]:
retrieval_cases = [
    ("What is LoRA?", "doc_lora"),
    ("What does recall at k measure?", "doc_metrics"),
    ("What is an LLM agent?", "doc_agents"),
    ("Why is causal masking used?", "doc_transformer"),
]

def recall_at_k(cases, k):
    hits = []
    for query, relevant in cases:
        result = retrieve(query, top_k=k)
        hits.append(relevant in set(result["document_id"]))
    return float(np.mean(hits))

def mean_reciprocal_rank(cases):
    values = []
    for query, relevant in cases:
        result = retrieve(query, top_k=len(chunk_frame))
        ranking = result["document_id"].tolist()
        rank = next(
            (i for i, doc in enumerate(ranking, start=1) if doc == relevant),
            None,
        )
        values.append(0.0 if rank is None else 1.0 / rank)
    return float(np.mean(values))

retrieval_metrics = pd.Series({
    "Recall@1": recall_at_k(retrieval_cases, 1),
    "Recall@3": recall_at_k(retrieval_cases, 3),
    "MRR": mean_reciprocal_rank(retrieval_cases),
})
retrieval_metrics

# 17. Task Success

In [ ]:
task_rows = []

for item in evaluation_cases:
    response = assistant_app(item["query"])
    success = item["must_contain"].lower() in response.answer.lower()
    task_rows.append({
        "query": item["query"],
        "answer": response.answer,
        "success": success,
    })

task_frame = pd.DataFrame(task_rows)
task_success = float(task_frame["success"].mean())
task_frame

# 18. Groundedness and Citation Validity

In [ ]:
def groundedness(answer, citations):
    if not citations:
        return 0.0

    evidence = []
    for citation in citations:
        match = chunk_frame[chunk_frame["chunk_id"] == citation]
        if len(match):
            evidence.append(match.iloc[0]["text"])

    if not evidence:
        return 0.0

    evidence_terms = set(normalize_text(" ".join(evidence)).split())
    answer_terms = [
        token for token in normalize_text(answer).split()
        if len(token) > 2
    ]

    if not answer_terms:
        return 0.0

    return sum(token in evidence_terms for token in answer_terms) / len(answer_terms)

def citation_validity(citations, retrieved):
    if not citations:
        return 0.0
    valid_ids = set(retrieved["chunk_id"])
    return sum(c in valid_ids for c in citations) / len(citations)

ground_rows = []
for query, _ in retrieval_cases:
    response = assistant_app(query)
    retrieved = retrieve(query, top_k=3)
    ground_rows.append({
        "query": query,
        "groundedness": groundedness(response.answer, response.citations),
        "citation_validity": citation_validity(response.citations, retrieved),
    })

ground_frame = pd.DataFrame(ground_rows)
ground_frame

# 19. Robustness

In [ ]:
robustness_queries = [
    "What does recall at k measure?",
    "Explain recall at k.",
    "What information does recall at k tell us?",
]

robustness_rows = []
for query in robustness_queries:
    response = assistant_app(query)
    robustness_rows.append({
        "query": query,
        "answer": response.answer,
        "confidence": response.confidence,
        "citations": response.citations,
    })

pd.DataFrame(robustness_rows)

# 20. Ablation

Compare sparse-only retrieval with the hybrid retriever.

In [ ]:
def sparse_only_retrieve(query, top_k=3):
    scores = sparse_scores(query)
    indices = np.argsort(scores)[::-1][:top_k]
    result = chunk_frame.iloc[indices].copy()
    result["score"] = scores[indices]
    return result

ablation_rows = []
for query, relevant in retrieval_cases:
    sparse_result = sparse_only_retrieve(query, 3)
    hybrid_result = retrieve(query, 3)
    ablation_rows.append({
        "query": query,
        "sparse_hit": relevant in set(sparse_result["document_id"]),
        "hybrid_hit": relevant in set(hybrid_result["document_id"]),
    })

ablation_frame = pd.DataFrame(ablation_rows)
ablation_frame

# 21. Error Taxonomy

In [ ]:
error_taxonomy = pd.DataFrame(
    [
        ("Intent error", "wrong route"),
        ("Retrieval miss", "relevant evidence absent"),
        ("Ranking error", "relevant evidence ranked too low"),
        ("Grounding error", "unsupported answer"),
        ("Citation error", "citation does not support answer"),
        ("Tool error", "tool call fails"),
        ("Schema error", "structured output invalid"),
        ("Abstention error", "system answers without sufficient evidence"),
    ],
    columns=["Error", "Description"],
)
error_taxonomy

# 22. System Card

## System
Offline NLP Research Assistant.

## Intended Use
Educational demonstration of end-to-end LLM-era NLP architecture.

## Capabilities
- retrieval QA;
- arithmetic;
- structured JSON;
- citations;
- confidence;
- abstention;
- limited Arabic support.

## Limitations
- deterministic generator rather than production LLM;
- small local corpus;
- latent SVD representation rather than neural embeddings;
- narrow rule-based intent router;
- no external web or enterprise data.

## Safety Design
- no side-effecting tools;
- explicit abstention;
- local-only data;
- source-linked answers.

# 23. Deployment and Safety

A production version would commonly replace:

- deterministic generation with an instruction-tuned LLM;
- SVD with neural embeddings;
- dataframe storage with a vector database;
- rule routing with a trained or LLM-based router;
- heuristic confidence with calibrated reliability estimation.

Safety and evaluation controls should remain after these replacements.

In [ ]:
summary = pd.Series({
    "Intent accuracy": intent_accuracy,
    "Retrieval Recall@3": recall_at_k(retrieval_cases, 3),
    "Retrieval MRR": mean_reciprocal_rank(retrieval_cases),
    "Task success": task_success,
    "Mean groundedness": float(ground_frame["groundedness"].mean()),
    "Mean citation validity": float(ground_frame["citation_validity"].mean()),
})

summary

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(summary.index, summary.values)
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("Lesson 50 Capstone Evaluation")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# 24. Knowledge Check

1. Why separate routing from retrieval?
2. Why combine sparse and semantic retrieval?
3. What does Recall@k measure?
4. What does MRR reward?
5. Why should groundedness be evaluated separately?
6. Why preserve citation identifiers?
7. What is abstention?
8. Why is confidence not automatically probability of correctness?
9. When is JSON output preferable?
10. Why should Arabic tashkeel be preserved in fully vocalized tasks?
11. What does an ablation study reveal?
12. Which capstone components are educational approximations?
13. What changes when replacing the deterministic generator with an LLM?
14. Why should safety controls remain?
15. Why should retrieval and answer-generation errors be separated?

# 25. Exercises

## Exercise 1
Expand the knowledge base to at least 25 documents.

## Exercise 2
Add query expansion and acronym handling.

## Exercise 3
Add a reranker.

## Exercise 4
Replace SVD with a sentence-embedding model.

## Exercise 5
Train a small intent classifier.

## Exercise 6
Add a second safe tool.

## Exercise 7
Add an entity-extraction JSON schema.

## Exercise 8
Calibrate confidence on a validation set.

## Exercise 9
Build a fully vocalized Arabic knowledge subset.

## Exercise 10
Write a final capstone report containing architecture, metrics, ablations, failures,
limitations, and future work.

## Challenge Exercises

1. Replace the deterministic answerer with a small local causal LM.
2. Add multilingual English–Arabic retrieval.
3. Add conversational memory with explicit scope.
4. Add a vector database.
5. Build a separate Streamlit or Gradio interface.

# 26. Course Summary

This capstone integrates the course progression from linguistic foundations through
classical NLP, embeddings, neural networks, Transformers, pretrained models,
fine-tuning, PEFT, LLM foundations, prompting, RAG, advanced retrieval, reliability,
agents, and multimodal foundations.

## Course Completion

**Lesson 50 completes Module 8: Large Language Models. The course continues with Module 9: Machine Translation.**

A separate advanced track can continue with:

- LLM deployment and inference optimization;
- quantization;
- distributed training;
- preference optimization;
- production RAG;
- advanced agents;
- observability;
- benchmark design;
- domain-specific capstones.

# Reproducibility Record

In [ ]:
reproducibility = pd.Series({
    "course": "Natural Language Processing: From Foundations to Large Language Models",
    "lesson": "Lesson 50",
    "documents": len(documents),
    "chunks": len(chunk_frame),
    "latent_components": n_components,
    "seed": SEED,
    "abstention_threshold": ABSTENTION_THRESHOLD,
    "python": platform.python_version(),
    "offline_execution": True,
})
reproducibility

# References

- Vaswani, A. et al. *Attention Is All You Need*.
- Lewis, P. et al. *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*.
- Karpukhin, V. et al. *Dense Passage Retrieval for Open-Domain Question Answering*.
- Hu, E. et al. *LoRA: Low-Rank Adaptation of Large Language Models*.
- Brown, T. et al. *Language Models are Few-Shot Learners*.
- Yao, S. et al. *ReAct: Synergizing Reasoning and Acting in Language Models*.
- Radford, A. et al. *Learning Transferable Visual Models From Natural Language Supervision*.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.